# Lab 10 — Midsem Questions 2, 4, 5 and BERT Fine-Tuning for Language Identification
**Introduction to LLM (AI463), SVNIT Surat**

1. **Part A** solves questions 2 (beam search), 4 (BPE) and 5 (TF-IDF and Doc2Vec) of the mid-semester paper. Each answer is worked out in the text and then checked with a few lines of code.
2. **Part B** fine-tunes a BERT-type model on the Lab 1 language-identification dataset and compares it with logistic regression, MLP, LSTM, GRU and a FastText-style classifier, all trained on the same train/test split.

# Part A — Midsem questions

In [5]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [6]:
import math
import pandas as pd
from collections import Counter

## Question 2 — Beam search with k = 2

**t = 1** — expand `<s>`: cat 0.5, sat 0.3, mat 0.2. Keep the top 2: **cat (0.5)** and **sat (0.3)**. `mat` is dropped.

**t = 2** — expand both kept hypotheses:

| Hypothesis | Cumulative probability |
|---|---|
| cat sat | 0.5 × 0.6 = **0.30** |
| cat mat | 0.5 × 0.3 = 0.15 |
| cat `</s>` | 0.5 × 0.1 = 0.05 |
| sat cat | 0.3 × 0.2 = 0.06 |
| sat mat | 0.3 × 0.7 = **0.21** |
| sat `</s>` | 0.3 × 0.1 = 0.03 |

Top 2 kept: **cat sat (0.30)** and **sat mat (0.21)**.

**t = 3** — expand both kept hypotheses:

| Hypothesis | Cumulative probability |
|---|---|
| cat sat mat | 0.30 × 0.8 = **0.24** |
| cat sat `</s>` | 0.30 × 0.2 = 0.06 |
| sat mat cat | 0.21 × 0.1 = 0.021 |
| sat mat `</s>` | 0.21 × 0.9 = **0.189** |

Top 2 kept: **cat sat mat (0.24)** and **sat mat `</s>` (0.189)**.

**Final output of the beam search:** `cat sat mat` with probability 0.24 (the best finished hypothesis ending in `</s>` is `sat mat </s>` with 0.189).

**Why P(cat | mat) and P(sat | mat) are ignored:** they would only be needed to expand the hypothesis `mat`, but `mat` (0.2) was pruned at t = 1 because only the top k = 2 hypotheses (cat 0.5 and sat 0.3) survive. A pruned hypothesis is never expanded, so its conditional probabilities are never used.

In [7]:
# Conditional probabilities given in the question: P(next word | words so far)
probs = {
    ("<s>",):       {"cat": 0.5, "sat": 0.3, "mat": 0.2},
    ("cat",):       {"sat": 0.6, "mat": 0.3, "</s>": 0.1},
    ("sat",):       {"cat": 0.2, "mat": 0.7, "</s>": 0.1},
    ("cat", "sat"): {"mat": 0.8, "</s>": 0.2},
    ("sat", "mat"): {"cat": 0.1, "</s>": 0.9},
}

def beam_search(probs, k=2, steps=3):
    beam = [((), 1.0)]                              # (words so far, cumulative probability)
    for t in range(1, steps + 1):
        candidates = []
        for words, score in beam:
            if words and words[-1] == "</s>":       # a finished hypothesis is carried forward unchanged
                candidates.append((words, score))
                continue
            key = words if words else ("<s>",)
            for next_word, p in probs.get(key, {}).items():
                candidates.append((words + (next_word,), score * p))

        candidates.sort(key=lambda c: c[1], reverse=True)
        print(f"t={t}: expanded hypotheses")
        for words, score in candidates:
            print("   ", " ".join(words), "->", round(score, 4))
        beam = candidates[:k]
        print("   kept:", [(" ".join(w), round(s, 4)) for w, s in beam], "\n")
    return beam

final_beam = beam_search(probs, k=2, steps=3)
best_words, best_score = final_beam[0]
print("Final output:", " ".join(best_words), "with probability", round(best_score, 4))

t=1: expanded hypotheses
    cat -> 0.5
    sat -> 0.3
    mat -> 0.2
   kept: [('cat', 0.5), ('sat', 0.3)] 

t=2: expanded hypotheses
    cat sat -> 0.3
    sat mat -> 0.21
    cat mat -> 0.15
    sat cat -> 0.06
    cat </s> -> 0.05
    sat </s> -> 0.03
   kept: [('cat sat', 0.3), ('sat mat', 0.21)] 

t=3: expanded hypotheses
    cat sat mat -> 0.24
    sat mat </s> -> 0.189
    cat sat </s> -> 0.06
    sat mat cat -> 0.021
   kept: [('cat sat mat', 0.24), ('sat mat </s>', 0.189)] 

Final output: cat sat mat with probability 0.24


## Question 4 — BPE with 2 merges

**Word frequencies:** low 5, lower 2, lowest 3, new 2, newest 1, widest 3. Each word is split into characters and an end-of-word marker `</w>` is added.

**Initial vocabulary:** l, o, w, e, r, s, t, n, i, d, `</w>`

**Pair counts:** (l, o) = 10, (o, w) = 10, (w, `</w>`) = 7, (e, s) = 7, (s, t) = 7, (t, `</w>`) = 7, (w, e) = 6, (n, e) = 3, (e, w) = 3, (w, i) = 3, (i, d) = 3, (d, e) = 3, (e, r) = 2, (r, `</w>`) = 2.

* **Merge 1:** (l, o) → `lo`. (l, o) and (o, w) tie at 10; the tie is broken by first occurrence in the corpus. Merging (o, w) first would give the same token `low` after two merges.
* **Merge 2:** (lo, w) → `low` (count 10).

**Merge rules:** 1. `l + o → lo`  2. `lo + w → low`

**Vocabulary after 2 merges (13 symbols):** l, o, w, e, r, s, t, n, i, d, `</w>`, lo, low

**Corpus after the merges:** `low </w>` × 5, `low e r </w>` × 2, `low e s t </w>` × 3, `n e w </w>` × 2, `n e w e s t </w>` × 1, `w i d e s t </w>` × 3

**Out-of-vocabulary words:** BPE splits an unseen word into characters and then applies the learned merge rules in the order they were learned. Whatever cannot be merged stays as smaller pieces, down to single characters, so every word can be written with known symbols. For example, `slow` becomes `s`, `low`, `</w>`. Only a character never seen in training needs an `<unk>` symbol (or byte-level BPE).

In [8]:
corpus = """low lower low lower lowest
low lowest low lowest
low new
newest widest widest
new widest"""

word_counts = Counter(corpus.split())                 # keeps the order of first appearance
words = {tuple(w) + ("</w>",): c for w, c in word_counts.items()}
base_symbols = sorted({s for symbols in words for s in symbols})


def count_pairs(words):
    pairs = Counter()
    for symbols, count in words.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += count
    return pairs


def merge_pair(words, pair):
    merged = {}
    for symbols, count in words.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        merged[tuple(new_symbols)] = count
    return merged


print("Word frequencies:", dict(word_counts))
print("Initial vocabulary:", base_symbols, "\n")

merges = []
for step in range(1, 3):
    pairs = count_pairs(words)
    best_pair = max(pairs, key=pairs.get)             # first pair with the highest count
    print(f"Merge {step}: top pair counts {pairs.most_common(4)}")
    print(f"         merge {best_pair} -> {best_pair[0] + best_pair[1]}")
    words = merge_pair(words, best_pair)
    merges.append(best_pair)

print("\nMerge rules:", [f"{a} + {b} -> {a + b}" for a, b in merges])
print("Vocabulary:", base_symbols + [a + b for a, b in merges])
print("Corpus after the merges:")
for symbols, count in words.items():
    print("   ", " ".join(symbols), "x", count)


def bpe_tokenize(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:                               # apply the rules in the order they were learned
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i + 1] == b:
                symbols[i:i + 2] = [a + b]
            else:
                i += 1
    return symbols


print("\nHandling words, including ones not in the corpus:")
for w in ["low", "lowest", "slow", "newer"]:
    print("   ", w, "->", bpe_tokenize(w, merges))

Word frequencies: {'low': 5, 'lower': 2, 'lowest': 3, 'new': 2, 'newest': 1, 'widest': 3}
Initial vocabulary: ['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w'] 

Merge 1: top pair counts [(('l', 'o'), 10), (('o', 'w'), 10), (('w', '</w>'), 7), (('e', 's'), 7)]
         merge ('l', 'o') -> lo
Merge 2: top pair counts [(('lo', 'w'), 10), (('w', '</w>'), 7), (('e', 's'), 7), (('s', 't'), 7)]
         merge ('lo', 'w') -> low

Merge rules: ['l + o -> lo', 'lo + w -> low']
Vocabulary: ['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', 'lo', 'low']
Corpus after the merges:
    low </w> x 5
    low e r </w> x 2
    low e s t </w> x 3
    n e w </w> x 2
    n e w e s t </w> x 1
    w i d e s t </w> x 3

Handling words, including ones not in the corpus:
    low -> ['low', '</w>']
    lowest -> ['low', 'e', 's', 't', '</w>']
    slow -> ['s', 'low', '</w>']
    newer -> ['n', 'e', 'w', 'e', 'r', '</w>']


## Question 5 — TF-IDF and Doc2Vec

**Vocabulary (7 words):** is, language, learning, machine, powerful, processing, useful

**Formulas used:** TF = (count of the word in the document) / (number of words in the document); IDF = log₂(N / df) with N = 4 documents. Using another log base only rescales every value.

| Word | df | IDF |
|---|---|---|
| is | 4 | log₂(4/4) = 0 |
| language | 2 | log₂(4/2) = 1 |
| learning | 3 | log₂(4/3) = 0.4150 |
| machine | 2 | 1 |
| powerful | 1 | log₂(4/1) = 2 |
| processing | 1 | 2 |
| useful | 3 | 0.4150 |

Every document has 4 words and each word occurs once, so TF = 0.25 for every word that is present and TF-IDF = 0.25 × IDF.

| Document | is | language | learning | machine | powerful | processing | useful |
|---|---|---|---|---|---|---|---|
| D1 | 0 | 0 | 0.1038 | 0.25 | 0 | 0 | 0.1038 |
| D2 | 0 | 0 | 0.1038 | 0.25 | 0.5 | 0 | 0 |
| D3 | 0 | 0.25 | 0.1038 | 0 | 0 | 0 | 0.1038 |
| D4 | 0 | 0.25 | 0 | 0 | 0 | 0.5 | 0.1038 |

The word `is` appears in every document, so its TF-IDF is 0 everywhere.

**Representing the classes through the TF-IDF of the documents:** each document is a 7-dimensional TF-IDF vector, and a class is represented by the **centroid (mean) of the TF-IDF vectors of its documents**:

* Technology = mean(D1, D2) → learning 0.1038, machine 0.25, powerful 0.25, useful 0.0519, all others 0
* Language = mean(D3, D4) → language 0.25, learning 0.0519, processing 0.25, useful 0.1038, all others 0

A new document is turned into a TF-IDF vector and given the class whose centroid is most similar (for example by cosine similarity). Equivalently, the four rows can be used as input features of a classifier with the class as the label.

**Learning document representations with Doc2Vec:** the technique is **Doc2Vec (Paragraph Vector)**. The architecture below is the **PV-DM (Distributed Memory)** model: a vector for the document ID and the vectors of the context words are combined, and the network predicts the next word.

```
 Document ID  ──► [ Paragraph matrix D ] ──┐
 word w(t-2)  ──► [ Word matrix W ]      ──┤
 word w(t-1)  ──► [ Word matrix W ]      ──┼──► average / concatenate ──► [ Softmax over vocabulary ] ──► predict w(t)
 word w(t+1)  ──► [ Word matrix W ]      ──┘
```

The paragraph vector acts as a memory of the topic of the document and is trained together with the word vectors. The other variant, **PV-DBOW**, uses only the document vector to predict words sampled from the document (like Skip-gram).

In [9]:
documents = {
    "D1": ("machine learning is useful",     "Technology"),
    "D2": ("machine learning is powerful",   "Technology"),
    "D3": ("learning language is useful",    "Language"),
    "D4": ("language processing is useful",  "Language"),
}

vocabulary = sorted({word for text, _ in documents.values() for word in text.split()})
N = len(documents)
doc_freq = {w: sum(w in text.split() for text, _ in documents.values()) for w in vocabulary}
idf = {w: math.log2(N / doc_freq[w]) for w in vocabulary}

print("Vocabulary:", vocabulary, "\n")
print(pd.DataFrame({"df": doc_freq, "idf": idf}).round(4).T, "\n")

rows = {}
for name, (text, _) in documents.items():
    tokens = text.split()
    rows[name] = {w: (tokens.count(w) / len(tokens)) * idf[w] for w in vocabulary}
tfidf_table = pd.DataFrame(rows).T
print("TF-IDF of every word in every document:")
print(tfidf_table.round(4), "\n")

class_of = pd.Series({name: label for name, (_, label) in documents.items()})
centroids = tfidf_table.groupby(class_of).mean()
print("Class representation (centroid of the TF-IDF vectors of its documents):")
print(centroids.round(4))

Vocabulary: ['is', 'language', 'learning', 'machine', 'powerful', 'processing', 'useful'] 

      is  language  learning  machine  powerful  processing  useful
df   4.0       2.0     3.000      2.0       1.0         1.0   3.000
idf  0.0       1.0     0.415      1.0       2.0         2.0   0.415 

TF-IDF of every word in every document:
     is  language  learning  machine  powerful  processing  useful
D1  0.0      0.00    0.1038     0.25       0.0         0.0  0.1038
D2  0.0      0.00    0.1038     0.25       0.5         0.0  0.0000
D3  0.0      0.25    0.1038     0.00       0.0         0.0  0.1038
D4  0.0      0.25    0.0000     0.00       0.0         0.5  0.1038 

Class representation (centroid of the TF-IDF vectors of its documents):
             is  language  learning  machine  powerful  processing  useful
Language    0.0      0.25    0.0519     0.00      0.00        0.25  0.1038
Technology  0.0      0.00    0.1038     0.25      0.25        0.00  0.0519


# Part B — Fine-tuning BERT for language identification

The Lab 1 dataset is loaded from a file, split once into train and test sets, and every model is trained and tested on that same split. This way the comparison between BERT and the earlier models (logistic regression, MLP, LSTM, GRU, FastText) is fair. The earlier models are re-trained here in compact form so that the notebook is self-contained.

**Speed-ups for Mac used in this notebook**
* Training runs on the Apple GPU (PyTorch `mps` backend), and `PYTORCH_ENABLE_MPS_FALLBACK` lets any unsupported operation run on the CPU instead of crashing.
* BERT uses dynamic padding (each batch is padded only to its own longest sentence), a short `max_length`, and frozen word embeddings. The embedding table is the largest part of a multilingual model, so freezing it saves a lot of memory and optimizer time at almost no cost in accuracy for this task.
* Inference sorts sentences by length so batches contain little padding.
* The loss is accumulated on the GPU and read once per epoch, because reading it every step forces the GPU to wait.
* The FastText-style model is tiny, so it runs on the CPU, where embedding-bag layers are fastest. Sparse matrices are turned into dense batches only one batch at a time.

In [10]:
%pip install -q torch "transformers<4.50" datasets scikit-learn pandas numpy tqdm
# NOTE: "transformers<4.50" keeps BERT working with PyTorch 2.2.x. If you can upgrade
# PyTorch to >= 2.5 you may drop that pin. Restart the kernel after installing.

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
# Let PyTorch fall back to the CPU for any operation the Mac GPU (MPS) does not support yet.
# This has to be set before torch is imported.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

### Step 1: Settings
Only this cell normally needs editing. The Lab 1 dataset (`ai4bharat/IndicCorpV2`) is **streamed** from Hugging Face instead of being read from a file.

In [12]:
DATASET_NAME         = "ai4bharat/IndicCorpV2"   # streamed from Hugging Face (as in Lab 1)
SENTENCES_PER_LANG   = 1000                      # 1000 per language -> 800 train / 200 test after the 80/20 split
MAX_ROWS             = None      # e.g. 30000 to use a random subset and make everything faster

SEED = 42

# BERT-type model. Other choices that support Indian languages:
#   "google/muril-base-cased"          (MuRIL, trained on 17 Indian languages)
#   "bert-base-multilingual-cased"     (mBERT, smaller vocabulary)
#   "ai4bharat/IndicBERTv2-MLM-only"   (IndicBERT v2)
BERT_MODEL      = "xlm-roberta-base"
BERT_EPOCHS     = 2
BERT_BATCH_SIZE = 32
BERT_LR         = 3e-5
BERT_MAX_LENGTH = 64

BASE_EPOCHS     = 3      # epochs for the MLP, LSTM and GRU
BASE_BATCH_SIZE = 256

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")         # Apple GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

Using device: mps


### Step 2: Stream the dataset and split it into train and test sets

In [15]:
import re
from datasets import load_dataset
from tqdm.auto import tqdm

languages = [
    'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva',
    'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva',
    'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya',
]

# Split paragraphs into sentences on standard and Indic punctuation
split_pattern = re.compile(r'[।!?.\n]+')

def get_sentences(text):
    sentences = split_pattern.split(text)
    return [s.strip() for s in sentences if len(s.strip()) > 15]   # drop empty / very short noisy pieces

print(f"Streaming {SENTENCES_PER_LANG} sentences for each of {len(languages)} languages...")

rows = []
for lang_split in tqdm(languages, desc="Processing Languages"):
    try:
        stream = load_dataset(DATASET_NAME, split=lang_split, streaming=True)
    except Exception as e:
        print(f"  skipping {lang_split}: {e}")
        continue

    lang_sentences = []
    for sample in stream:
        for sent in get_sentences(sample['text']):
            lang_sentences.append(sent)
            if len(lang_sentences) == SENTENCES_PER_LANG:
                break
        if len(lang_sentences) == SENTENCES_PER_LANG:
            break

    rows.extend({"text": s, "language": lang_split} for s in lang_sentences)

df = pd.DataFrame(rows)
text_col, label_col = "text", "language"
print("Streamed rows:", len(df))

# ---- same cleaning and split as before ----
df = df[[text_col, label_col]].dropna()
df[text_col] = df[text_col].astype(str).str.strip()
df = df[df[text_col] != ""]
df = df.drop_duplicates(subset=text_col)                          # the same sentence must not land in both train and test
df = df[df[label_col].map(df[label_col].value_counts()) >= 2]     # a class needs two sentences to be split
if MAX_ROWS and len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)

texts = df[text_col].tolist()
encoder = LabelEncoder()
labels = encoder.fit_transform(df[label_col].astype(str))
NUM_CLASSES = len(encoder.classes_)

train_texts, test_texts, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=SEED)
y_train, y_test = np.asarray(y_train), np.asarray(y_test)

print(f"\n{len(texts)} sentences, {NUM_CLASSES} languages: {list(encoder.classes_)}")
print(f"Train: {len(train_texts)}   Test: {len(test_texts)}")

Streaming 1000 sentences for each of 15 languages...


Processing Languages:   0%|          | 0/15 [00:00<?, ?it/s]

Streamed rows: 15000

14614 sentences, 15 languages: ['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya']
Train: 11691   Test: 2923


In [16]:
results = []

def record(name, y_pred, seconds):
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    results.append({"Model": name, "Accuracy": round(acc, 4), "Macro F1": round(f1, 4),
                    "Train+predict time (s)": round(seconds, 1)})
    print(f"{name}: accuracy {acc:.4f}, macro F1 {f1:.4f}, time {seconds:.1f}s")

### Step 3: The earlier models, re-trained on the same split

**Logistic regression** uses TF-IDF of character 1-3-grams. The **MLP** uses the same TF-IDF features.

In [17]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), max_features=50_000,
                        sublinear_tf=True, dtype=np.float32)
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

start = time.time()
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_tfidf, y_train)
record("Logistic Regression", lr_model.predict(X_test_tfidf), time.time() - start)

Logistic Regression: accuracy 0.9408, macro F1 0.9421, time 3.1s


In [18]:
# One training loop and one prediction function are shared by the MLP, LSTM, GRU and FastText-style models.
# get_batch(idx) returns ([model inputs], targets) for the rows listed in idx.

def fit_torch(model, get_batch, n_train, epochs, lr, batch_size, device):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(1, epochs + 1):
        model.train()
        order = np.random.permutation(n_train)
        total_loss = torch.zeros((), device=device)      # accumulated on the device; read once per epoch
        for begin in range(0, n_train, batch_size):
            idx = order[begin:begin + batch_size]
            inputs, targets = get_batch(idx)
            inputs = [t.to(device) for t in inputs]
            loss = loss_fn(model(*inputs), targets.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.detach() * len(idx)
        print(f"  epoch {epoch}: loss {total_loss.item() / n_train:.4f}")


@torch.no_grad()
def predict_torch(model, get_batch, n, batch_size, device):
    model.eval()
    predictions = []
    for begin in range(0, n, batch_size):
        idx = np.arange(begin, min(begin + batch_size, n))
        inputs, _ = get_batch(idx)
        logits = model(*[t.to(device) for t in inputs])
        predictions.append(logits.argmax(dim=1).cpu())
    return torch.cat(predictions).numpy()


class MLP(nn.Module):
    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden, n_classes))

    def forward(self, x):
        return self.net(x)


def sparse_batches(X, y):
    def get_batch(idx):
        dense = torch.from_numpy(X[idx].toarray())       # only one batch is made dense at a time
        return [dense], torch.from_numpy(y[idx])
    return get_batch


set_seed(SEED)
start = time.time()
print("MLP")
mlp = MLP(X_train_tfidf.shape[1], 256, NUM_CLASSES)
fit_torch(mlp, sparse_batches(X_train_tfidf, y_train), X_train_tfidf.shape[0], BASE_EPOCHS, 1e-3, BASE_BATCH_SIZE, DEVICE)
pred = predict_torch(mlp, sparse_batches(X_test_tfidf, y_test), X_test_tfidf.shape[0], BASE_BATCH_SIZE, DEVICE)
record("MLP", pred, time.time() - start)
del mlp

MLP
  epoch 1: loss 1.9019
  epoch 2: loss 0.5353
  epoch 3: loss 0.2350
MLP: accuracy 0.9456, macro F1 0.9468, time 5.1s


**LSTM and GRU** read the sentence character by character. Each character gets an embedding, the recurrent layer runs over the sentence, the outputs of the real (non-padding) characters are averaged, and a linear layer predicts the language.

In [19]:
MAX_CHARS = 100
chars = sorted({c for t in train_texts for c in t})
char_to_id = {c: i + 2 for i, c in enumerate(chars)}      # 0 = padding, 1 = unknown character


def encode_chars(texts):
    ids = np.zeros((len(texts), MAX_CHARS), dtype=np.int64)
    for row, t in enumerate(texts):
        codes = [char_to_id.get(c, 1) for c in t[:MAX_CHARS]]
        ids[row, :len(codes)] = codes
    return ids


train_ids, test_ids = encode_chars(train_texts), encode_chars(test_texts)


class CharRNN(nn.Module):
    def __init__(self, vocab_size, n_classes, cell="lstm", emb_dim=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        rnn_class = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn = rnn_class(emb_dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(hidden, n_classes)

    def forward(self, ids):
        mask = (ids != 0).unsqueeze(-1).float()                              # 1 for real characters, 0 for padding
        outputs, _ = self.rnn(self.embedding(ids))                           # (batch, length, hidden)
        pooled = (outputs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)  # average over real characters only
        return self.out(self.dropout(pooled))


def sequence_batches(ids, y):
    def get_batch(idx):
        batch = ids[idx]
        longest = int((batch != 0).sum(axis=1).max())                        # cut the padding that no sentence uses
        return [torch.from_numpy(batch[:, :longest])], torch.from_numpy(y[idx])
    return get_batch


for cell in ["lstm", "gru"]:
    set_seed(SEED)
    start = time.time()
    print(cell.upper())
    rnn_model = CharRNN(len(chars) + 2, NUM_CLASSES, cell=cell)
    fit_torch(rnn_model, sequence_batches(train_ids, y_train), len(train_ids), BASE_EPOCHS, 2e-3, BASE_BATCH_SIZE, DEVICE)
    pred = predict_torch(rnn_model, sequence_batches(test_ids, y_test), len(test_ids), BASE_BATCH_SIZE, DEVICE)
    record(cell.upper(), pred, time.time() - start)
    del rnn_model

LSTM
  epoch 1: loss 1.6237
  epoch 2: loss 0.7115
  epoch 3: loss 0.5699
LSTM: accuracy 0.8098, macro F1 0.8098, time 3.9s
GRU
  epoch 1: loss 1.5369
  epoch 2: loss 0.6950
  epoch 3: loss 0.5137
GRU: accuracy 0.8324, macro F1 0.8300, time 10.2s


**FastText-style classifier:** like FastText, a sentence is represented by the average of the embeddings of its hashed character n-grams (here 1 to 4 characters), and a single linear layer predicts the language. It is written in PyTorch so that it installs and runs easily on a Mac.

In [20]:
NUM_BUCKETS = 2 ** 17
hasher = HashingVectorizer(analyzer="char_wb", ngram_range=(1, 4), n_features=NUM_BUCKETS,
                           alternate_sign=False, norm="l1", dtype=np.float32)   # l1 norm: weights sum to 1, so the sum below is an average
X_train_hash = hasher.transform(train_texts)
X_test_hash = hasher.transform(test_texts)


class FastTextStyle(nn.Module):
    def __init__(self, n_buckets, n_classes, dim=64):
        super().__init__()
        self.embedding = nn.EmbeddingBag(n_buckets, dim, mode="sum")
        self.out = nn.Linear(dim, n_classes)

    def forward(self, indices, offsets, weights):
        return self.out(self.embedding(indices, offsets, per_sample_weights=weights))


def bag_batches(X, y):
    def get_batch(idx):
        batch = X[idx]
        indices = torch.from_numpy(batch.indices.astype(np.int64))      # n-gram ids of all sentences in the batch
        offsets = torch.from_numpy(batch.indptr[:-1].astype(np.int64))  # where each sentence starts
        weights = torch.from_numpy(batch.data.astype(np.float32))       # weight of each n-gram
        return [indices, offsets, weights], torch.from_numpy(y[idx])
    return get_batch


CPU = torch.device("cpu")      # a tiny model with an embedding bag runs fastest on the CPU
set_seed(SEED)
start = time.time()
print("FastText-style")
fasttext_model = FastTextStyle(NUM_BUCKETS, NUM_CLASSES)
fit_torch(fasttext_model, bag_batches(X_train_hash, y_train), X_train_hash.shape[0], 5, 1e-2, BASE_BATCH_SIZE, CPU)
pred = predict_torch(fasttext_model, bag_batches(X_test_hash, y_test), X_test_hash.shape[0], BASE_BATCH_SIZE, CPU)
record("FastText-style", pred, time.time() - start)
del fasttext_model

FastText-style
  epoch 1: loss 1.5655
  epoch 2: loss 0.3625
  epoch 3: loss 0.1651
  epoch 4: loss 0.0994
  epoch 5: loss 0.0613
FastText-style: accuracy 0.9459, macro F1 0.9472, time 3.4s


### Step 4: Fine-tune the BERT model
The sentences are tokenised once. Each batch is padded only to its own longest sentence, and the word-embedding table is frozen.

In [21]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def tokenize(texts):
    return tokenizer(texts, truncation=True, max_length=BERT_MAX_LENGTH)["input_ids"]

train_enc = tokenize(train_texts)
test_enc = tokenize(test_texts)


def bert_batch(encodings, y, idx):
    batch_ids = [encodings[i] for i in idx]
    longest = max(len(ids) for ids in batch_ids)
    input_ids = torch.full((len(batch_ids), longest), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch_ids), longest), dtype=torch.long)
    for row, ids in enumerate(batch_ids):
        input_ids[row, :len(ids)] = torch.tensor(ids)
        attention_mask[row, :len(ids)] = 1
    return input_ids, attention_mask, torch.from_numpy(y[idx])

In [22]:

set_seed(SEED)
bert = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=NUM_CLASSES)

# Freeze the word-embedding table: it is the biggest part of a multilingual model and does not need to change
for p in bert.base_model.embeddings.word_embeddings.parameters():
    p.requires_grad = False
bert.to(DEVICE)

trainable = [p for p in bert.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=BERT_LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_enc) / BERT_BATCH_SIZE)
total_steps = steps_per_epoch * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

start = time.time()
for epoch in range(1, BERT_EPOCHS + 1):
    bert.train()
    order = np.random.permutation(len(train_enc))
    total_loss = torch.zeros((), device=DEVICE)
    for step, begin in enumerate(range(0, len(order), BERT_BATCH_SIZE), start=1):
        idx = order[begin:begin + BERT_BATCH_SIZE]
        input_ids, attention_mask, targets = bert_batch(train_enc, y_train, idx)
        outputs = bert(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE),
                       labels=targets.to(DEVICE))
        optimizer.zero_grad()
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.detach() * len(idx)
        if step % 200 == 0:
            print(f"  epoch {epoch}: step {step}/{steps_per_epoch}, {time.time() - start:.0f}s elapsed")
    print(f"epoch {epoch}: average loss {total_loss.item() / len(order):.4f}")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  epoch 1: step 200/366, 252s elapsed
epoch 1: average loss 0.9161
  epoch 2: step 200/366, 811s elapsed
epoch 2: average loss 0.1948


In [23]:
@torch.no_grad()
def bert_predict(encodings, y, batch_size=128):
    bert.eval()
    order = np.argsort([len(e) for e in encodings])        # sort by length: little padding, faster batches
    predictions = np.zeros(len(encodings), dtype=np.int64)
    for begin in range(0, len(order), batch_size):
        idx = order[begin:begin + batch_size]
        input_ids, attention_mask, _ = bert_batch(encodings, y, idx)
        logits = bert(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE)).logits
        predictions[idx] = logits.argmax(dim=1).cpu().numpy()
    return predictions


record(f"BERT ({BERT_MODEL})", bert_predict(test_enc, y_test), time.time() - start)

del bert
if DEVICE.type == "mps":
    torch.mps.empty_cache()

BERT (xlm-roberta-base): accuracy 0.9384, macro F1 0.9396, time 1126.7s


### Step 5: Comparison of all models on the same test set

In [24]:
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

is_bert = results_df["Model"].str.startswith("BERT")
bert_row = results_df[is_bert].iloc[0]
best_other = results_df[~is_bert].iloc[0]
print(f"\nBest model overall: {results_df.iloc[0]['Model']} (accuracy {results_df.iloc[0]['Accuracy']})")
print(f"BERT accuracy {bert_row['Accuracy']} vs best earlier model {best_other['Model']} {best_other['Accuracy']}"
      f" (difference {bert_row['Accuracy'] - best_other['Accuracy']:+.4f})")

                  Model  Accuracy  Macro F1  Train+predict time (s)
         FastText-style    0.9459    0.9472                     3.4
                    MLP    0.9456    0.9468                     5.1
    Logistic Regression    0.9408    0.9421                     3.1
BERT (xlm-roberta-base)    0.9384    0.9396                  1126.7
                    GRU    0.8324    0.8300                    10.2
                   LSTM    0.8098    0.8098                     3.9

Best model overall: FastText-style (accuracy 0.9459)
BERT accuracy 0.9384 vs best earlier model FastText-style 0.9459 (difference -0.0075)
